# Stage 1 — fitting a euro government curve

The ECB fits a Svensson curve to AAA-rated euro area government bonds every business day, and publishes both the curve **and the parameters it fitted**. So we can do the whole exercise ourselves and then check our answer against the official one. That is an unusually good position to learn in — most of the time in finance nobody will tell you whether you got it right.

By the end of this notebook you should be able to:

1. move between discount factors, zero rates, par yields and forwards without hesitating,
2. say why anyone fits a parametric curve instead of interpolating,
3. write down Nelson–Siegel–Svensson from memory and explain what each parameter does,
4. name the one thing that most often makes an NSS fit go wrong.

In [ ]:
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from eurocurve import ecb
from eurocurve.nss import (
    NSSParams,
    fit_nss,
    nss_forward,
    nss_par_yield,
    nss_spot,
)

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

## 1. The data

The ECB Data Portal is a public SDMX API — no key, no signup. The yield curve dataset is `YC`, and a series key is a positional tuple of dimension codes:

```
B.U2.EUR.4F.G_N_A.SV_C_YM.SR_10Y
│ │  │   │  │     │       └── Spot Rate, 10 year
│ │  │   │  │     └────────── SVensson, Continuous compounding, Yield-error Minimisation
│ │  │   │  └──────────────── Government / Nominal / AAA issuers only (G_N_C = all issuers)
│ │  │   └─────────────────── ECB is the provider
│ │  └─────────────────────── euro
│ └────────────────────────── euro area, changing composition
└──────────────────────────── business daily
```

Two things about this data that will bite you if you skip them:

- **The API returns percent.** `2.5337` means 2.5337%, not 253%. `eurocurve.ecb` divides by 100 at the boundary so that nothing inside the package ever sees a percent.
- **The rates are continuously compounded.** So `DF = exp(-y*t)`, not `(1+y)^-t`. Get this wrong and you are out by about 3bp at 10 years — small enough to look plausible, large enough to matter.

In [ ]:
obs = ecb.fetch_curve()          # latest published day, AAA
as_of = obs.attrs['as_of']
print('as of', as_of)
obs.assign(rate_pct=lambda d: d['rate']*100).round(4)

## 2. Four ways to say the same thing

A yield curve is one function wearing four costumes. Learn to change between them instantly.

| name | definition | what it answers |
|---|---|---|
| discount factor `DF(t)` | `exp(-y(t)*t)` | what is a euro at time t worth now? |
| zero / spot rate `y(t)` | `-ln(DF(t))/t` | what single rate, compounded over t, gives that? |
| par yield `c(t)` | `(1-DF(t)) / sum_i DF(t_i)` | what coupon makes a t-year bond price at 100? |
| instantaneous forward `f(t)` | `-d ln DF / dt` | what rate is the market implying for an instant at time t? |

The relationship that ties the last one to the others is the one worth memorising:

```
y(T) = (1/T) * integral from 0 to T of f(u) du
```

**The spot rate is the average of the forwards.** Everything else follows. In particular this is why the forward curve is always more dramatic than the spot curve: averaging is smoothing, so the spot curve is a smoothed version of the forwards. When you see a gentle hump in spots, there is a violent one in forwards.

In [ ]:
t = obs['maturity'].values
y = obs['rate'].values
df = np.exp(-y*t)

tbl = pd.DataFrame({
    'tenor': obs['tenor'],
    'zero_cc_%': y*100,
    'zero_annual_%': (np.exp(y)-1)*100,
    'DF': df,
    'EUR100 in t is worth': 100*df,
})
tbl.round(4)

Note how small the continuous-vs-annual difference looks (a few basis points) and how large the discount factor effect is at 30 years. Both matter; they matter in different places.

## 3. Why not just interpolate?

The obvious move is to spline the 18 observed yields and be done. Let us see why nobody does that.

The test is never "does the zero curve look smooth" — every method passes that. The test is **what does the forward curve look like**, because the forwards are what you actually trade against and what a model calibration will consume.

In [ ]:
from scipy.interpolate import CubicSpline

spl = CubicSpline(t, y)
grid = np.linspace(t.min(), t.max(), 600)

# f(T) = y(T) + T*y'(T)  -- differentiate y(T)*T = integral of f
fwd_spline = spl(grid) + grid*spl(grid, 1)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(grid, spl(grid)*100); ax[0].plot(t, y*100, 'o', ms=5)
ax[0].set_title('cubic spline on zero rates — looks perfect')
ax[1].plot(grid, fwd_spline*100, color='crimson')
ax[1].set_title('...its implied forwards')
for a in ax: a.set_xlabel('years'); a.set_ylabel('%')
plt.tight_layout()

The forward curve wobbles. Those wiggles are not information — they are the spline reacting to a basis point of quoting noise at, say, the 6y point and amplifying it through the derivative. Nobody believes the market has a view that forward rates dip specifically at 6.3 years.

A parametric form fixes this by refusing to have that many degrees of freedom. Six parameters cannot chase noise, because six parameters cannot bend that many times.

## 4. Nelson–Siegel–Svensson

Nelson & Siegel (1987) wrote down a shape for the **forward** curve as a sum of exponential decay terms. Svensson (1994) added a second hump. The result:

```
f(t) = b0  +  b1*exp(-t/L1)  +  b2*(t/L1)*exp(-t/L1)  +  b3*(t/L2)*exp(-t/L2)
```

Now integrate and divide by T to get the spot curve. Define the workhorse

```
g(t, L) = (1 - exp(-t/L)) / (t/L)
```

and you get

```
y(t) = b0  +  b1*g(t,L1)  +  b2*(g(t,L1) - exp(-t/L1))  +  b3*(g(t,L2) - exp(-t/L2))
```

Six parameters, and each one means something you can say out loud:

| parameter | reads as |
|---|---|
| `b0` | the level the curve settles at for very long maturities, `y(inf)` |
| `b1` | how far the short end sits below (or above) that level; `b0+b1 = y(0)` |
| `b2` | the height of a medium-maturity hump — negative gives a trough |
| `b3` | a second hump, usually further out |
| `L1` | where the first hump peaks (at `t = L1`) |
| `L2` | where the second one peaks |

Two limits worth carrying around: at `t → 0`, `g → 1` and both hump terms vanish, so `y(0) = b0 + b1`. At `t → ∞` everything decays, so `y(∞) = b0`.

Here is what the four building blocks look like.

In [ ]:
L1, L2 = 2.0, 10.0
g = lambda tt, L: (1-np.exp(-tt/L))/(tt/L)
gr = np.linspace(0.01, 30, 500)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(gr, np.ones_like(gr), label='b0 loading — level')
ax[0].plot(gr, g(gr, L1), label='b1 loading — slope')
ax[0].plot(gr, g(gr, L1)-np.exp(-gr/L1), label=f'b2 loading — hump @ L1={L1}')
ax[0].plot(gr, g(gr, L2)-np.exp(-gr/L2), label=f'b3 loading — hump @ L2={L2}')
ax[0].set_title('loadings on the SPOT curve'); ax[0].legend()

ax[1].plot(gr, np.ones_like(gr))
ax[1].plot(gr, np.exp(-gr/L1))
ax[1].plot(gr, (gr/L1)*np.exp(-gr/L1))
ax[1].plot(gr, (gr/L2)*np.exp(-gr/L2))
ax[1].set_title('loadings on the FORWARD curve')
for a in ax: a.set_xlabel('years')
plt.tight_layout()

Look at the right panel: the `b2` loading peaks exactly at `t = L1` and the `b3` loading at `t = L2`. That is the whole reason the taus are called *decay* or *location* parameters. If you ever need to move a hump on a fitted curve, you change a tau, not a beta.

## 5. Fitting — and the one thing that goes wrong

We minimise squared error **in yields** (that is the `YM` in the ECB's `SV_C_YM`). The alternative, minimising price error, would weight long bonds enormously, since a 30-year bond's price moves ~20x more per basis point than a 2-year's.

Here is the failure mode to know about. Hold `L1` and `L2` fixed and the four betas enter **linearly** — the fit is just OLS, convex, one answer. But the taus enter non-linearly, and the objective in `(L1, L2)` has **several local minima**. The classic disaster is `L1` and `L2` converging on each other, at which point the `b2` and `b3` loadings become nearly identical, the model is unidentified, and the optimiser reports large offsetting betas with a perfectly fine-looking curve.

Fix: multi-start from a grid of tau pairs, keep the best. `fit_nss` does 40 starts by default. Cheap insurance.

In [ ]:
params, info = fit_nss(t, y)
print(params)
print(f"rmse      {info['rmse']*1e4:8.3f} bp")
print(f"max error {info['max_abs_error']*1e4:8.3f} bp")
print(f"y(0)  implied overnight = {params.short_rate:.4%}")
print(f"y(inf) implied long run = {params.long_rate:.4%}")

In [ ]:
gr = np.linspace(0.02, 30, 600)
fig, ax = plt.subplots(2, 1, figsize=(9, 8), sharex=True,
                       gridspec_kw={'height_ratios': [3, 1]})
ax[0].plot(t, y*100, 'o', ms=6, label='ECB observed', zorder=3)
ax[0].plot(gr, nss_spot(gr, params)*100, lw=2, label='NSS spot')
ax[0].plot(gr, nss_forward(gr, params)*100, '--', label='NSS instantaneous forward')
ax[0].axhline(params.beta0*100, ls=':', color='grey', label='b0')
ax[0].set_ylabel('%'); ax[0].legend(); ax[0].set_title(f'euro AAA curve, {as_of}')

ax[1].bar(t, info['residuals']*1e4, width=0.6)
ax[1].axhline(0, color='k', lw=0.8)
ax[1].set_ylabel('residual (bp)'); ax[1].set_xlabel('years')
plt.tight_layout()

Two things to notice in the residual panel.

First, the residuals are **not** random noise — they have structure, usually a pattern across the short end. That is NSS telling you honestly that six parameters cannot fit every wiggle. A bootstrapped curve would have zero residuals; that is the trade you made.

Second, the forward curve (dashed) always crosses the spot curve at the spot curve's turning points. That is not a coincidence: `y'(T) = (f(T) - y(T))/T`, so the spot curve is flat exactly where the forward crosses it. Handy for eyeballing whether a chart is self-consistent.

## 6. Grading ourselves against the ECB

This is the part you rarely get. The ECB publishes its own fitted `BETA0..BETA3`, `TAU1`, `TAU2` for the same day.

In [ ]:
official = ecb.fetch_ecb_svensson_params()
ecb_p = NSSParams(official['beta0'], official['beta1'], official['beta2'],
                  official['beta3'], official['tau1'], official['tau2'])

cmp = pd.DataFrame({
    'ours': params.as_array(),
    'ECB':  ecb_p.as_array(),
}, index=['beta0','beta1','beta2','beta3','tau1','tau2'])
cmp['diff'] = cmp['ours'] - cmp['ECB']
print(f"ECB parameters as of {official['as_of']}")
cmp.round(6)

In [ ]:
gr = np.linspace(0.25, 30, 400)
diff_bp = (np.asarray(nss_spot(gr, params)) - np.asarray(nss_spot(gr, ecb_p)))*1e4

plt.plot(gr, diff_bp)
plt.axhline(0, color='k', lw=0.8)
plt.xlabel('years'); plt.ylabel('our fit minus ECB (bp)')
plt.title(f'curve difference — max {np.abs(diff_bp).max():.2f} bp')
plt.show()

**Read that carefully.** The parameters may differ noticeably. The *curves* agree to a fraction of a basis point.

This is the practical face of NSS being weakly identified: many parameter vectors produce nearly the same curve, so the optimiser's choice among them is close to arbitrary. Consequences you should carry forward:

- **Never compare betas across dates or countries** as if they were economic quantities. People publish papers regressing things on `b2` and it is mostly noise in the fit.
- **Always validate on the curve**, not the coefficients. Our test suite does exactly this.
- If you need stable parameters over time — for a time-series study, say — fix `L1` and `L2` at constants and only refit the betas. You lose a little fit quality and gain interpretable coefficients.

## 7. A second, independent check: forwards and par yields

The ECB publishes instantaneous forwards (`IF_10Y`) and par yields (`PY_10Y`) too. Our fitted parameters should reproduce both — and crucially, these come from *different* closed-form expressions in our code, so agreement means the spot/forward/par-yield formulas are mutually consistent, not just that we copied one formula correctly.

In [ ]:
obs_f  = ecb.fetch_curve(kind='IF')
obs_py = ecb.fetch_curve(kind='PY')

chk = pd.DataFrame({
    'tenor': obs_f['tenor'],
    'ECB fwd %':  obs_f['rate'].values*100,
    'ours fwd %': np.asarray(nss_forward(obs_f['maturity'].values, params))*100,
    'ECB par %':  obs_py['rate'].values*100,
    'ours par %': np.asarray(nss_par_yield(obs_py['maturity'].values, params, freq=1))*100,
})
chk['fwd err bp'] = (chk['ours fwd %']-chk['ECB fwd %'])*100
chk['par err bp'] = (chk['ours par %']-chk['ECB par %'])*100
chk.round(3)

The forward errors should be tiny. The par yield errors will be a bit larger — the ECB's par yield definition uses its own coupon frequency and day-count assumptions which we have simplified to annual, act/act-ish. Worth knowing that a "par yield" is not fully defined until you state the coupon frequency.

## 8. Feeling the parameters

Perturb one beta at a time and watch what moves. Do this once and you will never have to look up what `b2` does again.

In [ ]:
gr = np.linspace(0.02, 30, 400)
base = params.as_array()
labels = ['beta0 (level)', 'beta1 (slope)', 'beta2 (hump 1)', 'beta3 (hump 2)']

fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True)
for i, (axx, lab) in enumerate(zip(axes.ravel(), labels)):
    axx.plot(gr, np.asarray(nss_spot(gr, base))*100, 'k', lw=2, label='fitted')
    for bump in (-0.005, 0.005):
        p = base.copy(); p[i] += bump
        axx.plot(gr, np.asarray(nss_spot(gr, p))*100, alpha=0.75,
                 label=f'{bump:+.1%}')
    axx.set_title(lab); axx.legend(fontsize=8)
plt.tight_layout()

## 9. AAA vs everyone — the curve as a credit measure

`G_N_A` is AAA issuers only; `G_N_C` is all euro area governments. The gap between them is a market-implied measure of euro area fragmentation. In calm periods it is 20–40bp; in 2011–12 it was hundreds.

In [ ]:
aaa = ecb.fetch_curve(rating=ecb.RATING_AAA)
allg = ecb.fetch_curve(rating=ecb.RATING_ALL)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(aaa['maturity'], aaa['rate']*100, 'o-', label='AAA only')
ax[0].plot(allg['maturity'], allg['rate']*100, 's-', label='all issuers')
ax[0].legend(); ax[0].set_title('spot curves'); ax[0].set_ylabel('%')
ax[1].plot(aaa['maturity'], (allg['rate'].values-aaa['rate'].values)*1e4, 'o-', color='crimson')
ax[1].set_title('all-issuer minus AAA (bp) — fragmentation premium')
for a in ax: a.set_xlabel('years')
plt.tight_layout()

In [ ]:
# how unusual is today? ten years of the 10y point
h = ecb.fetch_history('10Y', start='2015-01-01')
h.plot(figsize=(10,4))
plt.axhline(float(nss_spot(10.0, params)), color='crimson', ls='--', label='today')
plt.title('euro AAA 10y spot rate'); plt.ylabel('decimal'); plt.legend(); plt.show()
print(f"today is at the {100*(h < float(nss_spot(10.0, params))).mean():.0f}th percentile of the last decade")

## Exercises

Do these in this notebook. They are all short, and each one teaches something the reading cannot.

1. **Break the fit on purpose.** Call `fit_nss(..., n_starts=1)` and compare the fitted curve to the multi-start version across a month of dates. Count how often single-start lands somewhere worse. This is the argument for multi-start, made with your own data.

2. **Fit plain Nelson–Siegel** by forcing `beta3 = 0`. How much worse is the RMSE? Where on the curve does it fail? (Hint: the second hump usually earns its keep beyond 10 years.)

3. **Weight the fit.** Pass `weights=1/np.sqrt(t)` so the front end matters more. Does the 30y point deteriorate? By how much? Which weighting would you want if you were pricing a 2y note?

4. **Fix the taus.** Refit across 60 business days with `L1, L2` free, then with them pinned at their median values. Plot `beta2` over time both ways. The free version is much noisier — that is the identification problem showing up as a time series.

5. **Price something.** Take a hypothetical 10-year bond with a 3% annual coupon. Price it off your fitted curve. Now compute its modified duration numerically (bump every rate 1bp, reprice, divide). Sanity-check the answer against the rule of thumb that a par bond's duration is a bit under its maturity.

6. **Steepness.** Build a time series of `y(10) - y(2)` from `fetch_history` for both tenors. Overlay ECB policy rate decisions. The 2s10s slope is the single most-watched number in rates — see whether it led or lagged.

Then move on to `02_ois_bootstrap.ipynb`, which builds a curve the opposite way.